# Notebook 10 - Build the RAG regulation index

**Owner:** Member 4 (Infra + RAG) - **Runtime:** GPU (T4) - **Runs:** offline, never in a request path

Turns the CERC/IEGC PDFs into an embedded, searchable corpus in Postgres.

A T4 embeds a ~2000-chunk corpus in well under a minute. The same work on a laptop CPU
takes 10-20 minutes, which is why this is a notebook and not a step in the deploy.

## Two failure modes this notebook is built to avoid

Both are silent. They produce no error, just a copilot that cites confidently and wrongly.

1. **Embedding model mismatch.** If the corpus is embedded with one model and queries are
   embedded with another, the nearest neighbours are meaningless. Every row records its
   `embed_model`, and the server refuses to start when they disagree.
2. **An ANN index built on an empty table.** `ivfflat` clusters on whatever rows exist at
   build time; built empty, recall collapses. Insert first, index after - always. The
   schema uses HNSW, which is safe either way, but the ordering below is correct for both.

## Run order

Runtime -> Change runtime type -> **T4 GPU**, then Run all. Set `DATABASE_URL` in cell 3 first.


## 1. Dependencies


In [ ]:
!pip -q install pymupdf FlagEmbedding psycopg2-binary pgvector sqlalchemy

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - this will be slow')


## 2. Get the code and the corpus

Clone the repo so the chunker used here is byte-identical to the one the test suite
covers. Re-implementing the chunker in the notebook is how a notebook and a server drift
apart without anyone noticing.


In [ ]:
import os, pathlib

REPO = 'https://github.com/Meetvirugama/AI-Powered-Renewable-Generation-Forecasting-Platform.git'
BRANCH = 'main'

if not pathlib.Path('/content/platform').exists():
    !git clone -q --branch {BRANCH} {REPO} /content/platform

os.chdir('/content/platform')
import sys; sys.path.insert(0, '/content/platform')

# The PDFs are committed, so the corpus is reproducible from a clean clone.
!ls -la regulations/


## 3. Configuration

Point `DATABASE_URL` at RDS through an SSM port-forward:

```bash
aws ssm start-session --target <instance-id> \
  --document-name AWS-StartPortForwardingSessionToRemoteHost \
  --parameters '{"host":["<rds-endpoint>"],"portNumber":["5432"],"localPortNumber":["5432"]}'
```

Colab cannot reach a private RDS directly. Either run the insert cell as a script on the
EC2 box, or use the tunnel above. Do not make RDS publicly accessible to save time - that
is the one shortcut that turns into a security finding.


In [ ]:
from getpass import getpass

# getpass, not a literal: never leave a password in a cell you might share.
DATABASE_URL = getpass('DATABASE_URL: ')

EMBED_MODEL = 'BAAI/bge-m3'   # 1024-dim; must match RAG_EMBED_MODEL on the server
BATCH_SIZE  = 32
TRUNCATE    = True            # rebuild the corpus from scratch

os.environ['RAG_EMBED_MODEL'] = EMBED_MODEL
os.environ['RAG_EMBED_BACKEND'] = 'bge'


## 4. Chunk the PDFs

Clause-aware: split on legal structure first, then size-bound within a clause. A chunk
never spans two clauses, so the clause id attached to a chunk is the clause the text
actually came from - which is what makes the citation trustworthy.


In [ ]:
from backend.modules.rag import ingest

chunks = ingest.ingest_all(pathlib.Path('regulations'))
print(f'{len(chunks)} chunks')


## 5. Validate the chunker BEFORE embedding anything

**Do not skip this cell.** Embedding a badly chunked corpus is the classic wasted
afternoon, and nothing downstream will tell you it happened.

- `unmatched` above ~30% means the PDF uses a numbering style `CLAUSE_PATTERNS` does not
  cover. Add a pattern; do not lower the standard.
- Zero deviation-charge hits means the wrong corpus.
- An exception about extracted characters means an image-only PDF that needs OCR.


In [ ]:
import importlib.util

spec = importlib.util.spec_from_file_location('build_index', 'scripts/build_index.py')
build_index = importlib.util.module_from_spec(spec)
spec.loader.exec_module(build_index)

looks_ok = build_index.report(chunks)
assert looks_ok, 'chunker output looks wrong - fix it before spending GPU time'


## 6. Embed on the GPU

`use_fp16=True` is safe here and roughly doubles throughput. The server runs CPU-only in
fp32; the resulting vectors are compatible.


In [ ]:
from FlagEmbedding import BGEM3FlagModel
import numpy as np

model = BGEM3FlagModel(EMBED_MODEL, use_fp16=True)

texts = [c.chunk_text for c in chunks]
vectors = model.encode(texts, batch_size=BATCH_SIZE, max_length=1024)['dense_vecs']
vectors = np.asarray(vectors, dtype=np.float32)

print('shape:', vectors.shape)
assert vectors.shape[1] == 1024, 'dimension must match VECTOR(1024) in the schema'


## 7. Write to Postgres

Upsert keyed on `chunk_id`, so re-running this notebook updates the corpus rather than
duplicating it. `embed_model` is stored on every row - that column is what lets the
server detect a corpus/query mismatch in under a second at startup.


In [ ]:
from sqlalchemy import create_engine, text as sql

engine = create_engine(DATABASE_URL)

with engine.begin() as conn:
    conn.execute(sql('CREATE EXTENSION IF NOT EXISTS vector'))
    if TRUNCATE:
        conn.execute(sql('DELETE FROM regulation_chunks'))

    for chunk, vector in zip(chunks, vectors):
        conn.execute(sql('''
            INSERT INTO regulation_chunks
                (doc_name, section, clause, page_no, source_url, effective_date,
                 chunk_text, chunk_id, embedding, embed_model)
            VALUES (:doc_name, :section, :clause, :page_no, :source_url, :effective_date,
                    :chunk_text, :chunk_id, CAST(:embedding AS vector), :embed_model)
            ON CONFLICT (chunk_id) DO UPDATE SET
                chunk_text  = EXCLUDED.chunk_text,
                embedding   = EXCLUDED.embedding,
                embed_model = EXCLUDED.embed_model
        '''), {
            'doc_name': chunk.doc_name, 'section': chunk.section, 'clause': chunk.clause,
            'page_no': chunk.page_no, 'source_url': chunk.source_url,
            'effective_date': chunk.effective_date or None,
            'chunk_text': chunk.chunk_text, 'chunk_id': chunk.stable_id(),
            'embedding': '[' + ','.join(f'{float(v):.6f}' for v in vector) + ']',
            'embed_model': EMBED_MODEL,
        })

print('inserted', len(chunks))


## 8. Build the indexes - AFTER the rows exist

HNSW rather than ivfflat: no training step, no `lists` tuning, and no dependence on the
data distribution at build time. For a corpus under ~3k rows a sequential scan is about
5 ms anyway, so this is insurance rather than a necessity.


In [ ]:
with engine.begin() as conn:
    conn.execute(sql('''
        CREATE INDEX IF NOT EXISTS ix_regulation_chunks_embedding_hnsw
        ON regulation_chunks USING hnsw (embedding vector_cosine_ops)
        WITH (m = 16, ef_construction = 64)
    '''))
    conn.execute(sql('''
        CREATE INDEX IF NOT EXISTS ix_regulation_chunks_fts
        ON regulation_chunks USING gin (to_tsvector('english', chunk_text))
    '''))
print('indexes ready')


## 9. Verify

One `embed_model` value, a plausible chunk count, and a sane nearest neighbour.


In [ ]:
with engine.connect() as conn:
    print(conn.execute(sql(
        'SELECT embed_model, count(*) FROM regulation_chunks GROUP BY embed_model')).all())

    qv = model.encode(['deviation charges for a solar seller'], max_length=512)['dense_vecs'][0]
    literal = '[' + ','.join(f'{float(v):.6f}' for v in qv) + ']'
    rows = conn.execute(sql('''
        SELECT doc_name, clause, page_no,
               1 - (embedding <=> CAST(:qv AS vector)) AS score,
               left(chunk_text, 160) AS preview
        FROM regulation_chunks
        ORDER BY embedding <=> CAST(:qv AS vector)
        LIMIT 5'''), {'qv': literal}).all()

for r in rows:
    print(f'{r.score:.3f}  {r.doc_name} | {r.clause} | p.{r.page_no}')
    print(f'        {r.preview}...')


## 10. Score the retriever

The only objective signal anyone on this project gets about RAG quality. Target
**recall@5 >= 0.7**. Below that, fix the chunker - a prompt cannot cite a clause that
retrieval never surfaced.


In [ ]:
!python scripts/eval_retrieval.py --database-url "$DATABASE_URL" --verbose


---

## Checklist before closing the tab

- [ ] `SELECT embed_model, count(*) ... GROUP BY 1` returns **one** model and a plausible count
- [ ] The nearest-neighbour query returns a genuinely relevant clause
- [ ] `recall@5 >= 0.7`
- [ ] `curl $BACKEND/rag/health` shows the same chunk count and no `warning` field

If `/rag/health` reports a `warning`, the server is querying with a different model than
the one used here. Fix `RAG_EMBED_MODEL` on the server or re-run this notebook - do not
ship it. Retrieval will look like it works and will be wrong.
